# 09_validations - TO BE DISCUSSED LATER   [PLAN A]

Intentionally empty notebook: we will decide its content together. Tests considered so far, for the record:

1. **Downscaling alone**: F2 trained without 2015-2025 (`VALIDATION_MODE = True`, notebook step3), applied to ERA5 1 deg over 2015-2025, compared with ERA5 0.25 deg (maps, quantiles 50 / 95 / 99).
2. **F1 (+ F2) in past climate**: adjusted EC-Earth3 historical against ERA5 over the same period (dry-day fraction, drizzle, quantiles, maps).
3. **F1 out of sample**: training 1981-2014, test 2015-2025 (SSP5-8.5); and splits 1981-2003 / 2004-2025. It can only be reassuring (forcing differs from ERA5).
4. **GCM signal preservation**: 2025 - historical, and hosing - no-hosing, before / after F1 (same scale).
5. **F3 on two halves** of the RCM 2025 no-hosing run (25 / 25 years).
6. **Hosing delta before / after F3** + **range test**: per pixel and per month, bounds of the raw RCM no-hosing run (min/max or 0.5 / 99.5 percentiles), fraction of hosing days outside the bounds (upper bound only for `pr`), with the second half vs the first half of the no-hosing run as the baseline.
7. **Indicative comparison with ISIMIP3b EC-Earth3 (0.5 deg)**: sanity check only (W5E5 is not ERA5).
8. Possible extra per-pixel WDF+QDM pass at 0.25 deg after the downscaling, only if test 2 shows a residual marginal per-pixel bias.

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
BBOX = {"lat": tuple(cfg["europe_bbox"]["lat"]), "lon": tuple(cfg["europe_bbox"]["lon"])}

# TODO: to be written after discussion